# 02 — Isolation Forest, CUSUM ve Modüler Dedektör Füzyonu (Faz ML-1)

**Paradigma — yarı-gözetimli novelty detection**: model **yalnızca normal** uçuşlarla eğitilir.
Etiketler eğitime girmez; sadece değerlendirmede kullanılır. İlk hedef ikili: normal (0) / anomali (1).

Bu notebook üç aşamalı ilerler ve her aşamanın *neden* gerekli olduğunu deneyle gösterir:

1. **Monolitik IF** (tüm feature'lar, satır bazlı) → başarısız olur; *neden* başarısız olduğu bu verinin en öğretici dersidir.
2. **Etiket semantiği ve aile analizi** → her anomali ailesinin farklı sinyalde yaşadığı gösterilir.
3. **Modüler dedektörler + skor füzyonu** → FableChat mimarisi: aile başına dar feature'lı dedektör, uçuş düzeyi birleşim.

## 1. Isolation Forest matematiği

Fikir: anomaliler *az ve farklı* oldukları için rastgele bölmelerle **daha çabuk izole olur**.

Her ağaç rastgele bir feature ve rastgele bir kesim noktasıyla veriyi böler. Örneğin **yol uzunluğu** $h(x)$ =
kökten, örneğin tek başına kaldığı yaprağa kadarki bölme sayısı. $n$ örnekli düğümde başarısız BST aramasının
beklenen yolu:

$$c(n) = 2H(n-1) - \frac{2(n-1)}{n}, \qquad H(i) \approx \ln(i) + \gamma,\; \gamma \approx 0.5772$$

Anomali skoru, ağaçlar üzerinden ortalanmış yolun $c(n)$ ile normalize edilmişi:

$$s(x, n) = 2^{-\,\mathbb{E}[h(x)]\,/\,c(n)} \in (0,1]$$

- $\mathbb{E}[h(x)] \to 0$ (hemen izole) $\Rightarrow s \to 1$ → **anomali**
- $\mathbb{E}[h(x)] \approx c(n)$ (ortalama derinlik) $\Rightarrow s \approx 0.5$ → normal

`sklearn.score_samples` $\approx -s$ döndürür; **anomali_skoru $= -$score_samples** kullanırız (büyük = anormal).
`max_samples=256` (Liu et al.): alt-örnekleme *swamping/masking*'i azaltır ve $c(256) \approx 15.6$'yı sabitler.

**Feature sayısı tuzağı**: IF her bölmede feature'ı *rastgele* seçer. 58 feature'ın 3'ü bilgilendiriciyse,
ağaçların büyük kısmı gürültü boyutlarında bölünür → bilgilendirici sinyal **sulanır**. Bu, aşağıda deneyle görülecek
ve modüler tasarımın matematiksel gerekçesidir.

## 2. Threshold: contamination değil, normal persentili

$$\tau = Q_{0.99}\big(\{s(x) : x \in \text{val}_{normal}\}\big)$$

Tanım gereği beklenen normal yanlış-alarm oranı ≈ %1 ve $\tau$ hiçbir anomali bilgisi içermez
(tek jamming loguna göre eşik ayarlanamaz — o log testte kalmalı).

## 3. CUSUM — ısrarlı sapma biriktirici

$$S^+_t = \max(0,\, S^+_{t-1} + r_t - k), \qquad S^-_t = \max(0,\, S^-_{t-1} - r_t - k)$$

$k = 0.5\hat\sigma$, $\hat\sigma = 1.4826\cdot\mathrm{MAD}$ (uçuş-içi, robust). Tek sıçramayı IF, **yavaş ama
kalıcı** kaymayı CUSUM yakalar; alarm anı ürettiği için Detection-Time metriklerine doğal bağlanır.

## 4. Metrikler

| Metrik | Tanım | Not |
|---|---|---|
| Uçuş ROC-AUC | $P(\text{skor}_{anomalili\ uçuş} > \text{skor}_{normal\ uçuş})$ | asıl karşılaştırma ekseni |
| Satır ROC/PR-AUC | satır bazlı ayrım | *etiket semantiği uyarısıyla* (aşağıda) |
| Recall@τ (tip bazında) | anomali tipi başına yakalama | kapsam haritasına girer |
| FPR@τ | normal satır/uçuşta yanlış alarm | operasyonel maliyet |
| Avg/Max Detection Time | onset → ilk alarm gecikmesi | ALFA makalesinin resmî metrikleri |


In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

FEAT = ROOT / "data/gold/ml_features"
manifest = json.loads((FEAT / "split_manifest.json").read_text(encoding="utf-8"))

from src.ml.data.scaling import apply_scaler_params
from src.ml.features.temporal import cusum

NORMALS = {"normal", "benign"}

def load_source(name):
    df = pd.read_parquet(FEAT / name / f"{name}_ml_features.parquet")
    scaler = json.loads((ROOT / "artifacts/scalers" / f"{name}_robust_scaler.json").read_text())
    return df, scaler, scaler["feature_columns"]

def anomaly_scores(model, X):
    return -model.score_samples(X)

print("hazir")

hazir


## 5. Deney A — Monolitik IF (tüm feature'lar, satır bazlı)

Önce planın ilk hâlini olduğu gibi deniyoruz: tüm feature seti, satır bazlı skor, $\tau = Q_{0.99}$(val).

In [2]:
def run_monolithic(name):
    df, scaler, cols = load_source(name)
    labels = manifest["sources"][name]["flight_labels"]
    rows = []
    for split_name, split in manifest["sources"][name]["splits"].items():
        scaled = apply_scaler_params(df, scaler)
        tr = scaled[scaled["source_id"].isin(split["train"])]
        va = scaled[scaled["source_id"].isin(split["val"])]
        te = scaled[scaled["source_id"].isin(split["test"])]
        m = IsolationForest(n_estimators=300, max_samples=256,
                            random_state=split["seed"], n_jobs=-1).fit(tr[cols])
        tau = np.quantile(anomaly_scores(m, va[cols]), 0.99)
        s = anomaly_scores(m, te[cols])
        y = (~te["label"].isin(NORMALS)).astype(int).to_numpy()
        fs = te.assign(score=s).groupby("source_id")["score"].apply(lambda x: np.percentile(x, 95))
        yf = np.array([0 if labels[f] == "normal" else 1 for f in fs.index])
        rows.append({"split": split_name,
                     "satir_roc": roc_auc_score(y, s),
                     "satir_pr": average_precision_score(y, s),
                     "ucus_roc": roc_auc_score(yf, fs.values),
                     "fpr@tau": float((s[y == 0] > tau).mean())})
    return pd.DataFrame(rows).set_index("split")

for name in ["alfa", "uav_attack"]:
    r = run_monolithic(name)
    print(f"=== {name} — monolitik IF ===")
    print((r.mean().round(3).astype(str) + " +- " + r.std().round(3).astype(str)).to_string(), "\n")

=== alfa — monolitik IF ===
satir_roc    0.497 +- 0.026
satir_pr     0.971 +- 0.011
ucus_roc     0.614 +- 0.271
fpr@tau      0.038 +- 0.074 



=== uav_attack — monolitik IF ===
satir_roc    0.209 +- 0.138
satir_pr     0.937 +- 0.016
ucus_roc     0.415 +- 0.201
fpr@tau      0.211 +- 0.434 



### Deney A'nın dürüst okuması — üç yapısal sorun

Satır ROC ≈ 0.5 (ALFA) ve < 0.5 (UAV Attack): satır bazlı monolitik model **ayrıştırmıyor**. Nedenleri:

1. **Etiket semantiği**: Saldırı loglarının *tüm* satırları "anomali" etiketli, ama imza yalnızca birkaç satırda
   görünür (78 km'lik sıçrama ~2-3 satır). Satır bazlı ROC, imza taşımayan binlerce "anomali" satırını modelin
   yakalamasını bekler — **adaletsiz bir hedef**. PR-AUC'un yüksek görünmesi de aynı nedenle yanıltıcıdır: test
   satırlarının ~%90'ı anomali etiketli olduğundan şans çizgisi (prevalence) zaten ~0.9'dur.
2. **Feature sulanması**: 58 boyutun 3-4'ü bilgilendirici; IF bölmelerinin çoğu gürültüde harcanır (Bölüm 1'deki tuzak).
3. **Impute etkisi**: eksik değerler train medyanıyla doldurulur → attitude'u tamamen eksik DoS satırları
   "tam ortalama" görünür ve normal satırlardan bile derin (normal-benzeri) izole olur — UAV'de ROC'un
   0.5'in *altına* düşmesinin nedeni budur.

**Sonuç**: değerlendirme uçuş düzeyine, mimari aile-bazlı modüllere taşınmalı. Önce aileleri kanıtlayalım.

## 6. Deney B — Aile analizi: her anomali hangi sinyalde yaşıyor?

Uçuş başına özet istatistiklerle "hangi feature hangi aileyi ayırıyor" tablosu:

In [3]:
df_u, _, _ = load_source("uav_attack")
labels_u = manifest["sources"]["uav_attack"]["flight_labels"]
agg = df_u.groupby("source_id").agg(
    jamming_max=("jamming_5s_max", "max"), noise_std_max=("noise_5s_std", "max"),
    hdop_max=("hdop_5s_max", "max"), sats_min=("sats_5s_min", "min"),
    gps_step_max=("gps_step_5s_max", "max"), spd_residual_max=("gps_speed_residual", "max"),
    att_missing=("attitude_missing", "mean"))
agg["label"] = [labels_u[f] for f in agg.index]
print("UAV Attack — ucus medyanlari (aile imzalari):")
print(agg.groupby("label").median().round(2).to_string())
print()
print("Kritik gozlemler:")
print("- gps_jamming: jamming/noise/hdop yuksek, uydu sayisi dusuk -> SINYAL-KALITESI ailesi")
print("- gps_spoofing (SITL): 78 km konum sicramasi -> NAV-BUTUNLUGU ailesi (kaba)")
live = agg.loc[[i for i in agg.index if "hackrf" in i]]
print(f"- gps_spoofing (LIVE/hackrf): sicrama YOK (step={live['gps_step_max'].iloc[0]:.1f} m) ama")
print(f"  receiver-vs-konum hiz residual'i {live['spd_residual_max'].iloc[0]:.1f} (normal maks ~2) -> analytical redundancy GIZLI spoofing'i de yakaliyor")
dos = agg[agg["label"] == "ping_dos"]
print(f"- ping_dos: {int((dos['att_missing'] > 0.5).sum())}/6 logda attitude tamamen eksik (VERI-KALITESI ailesi);")
print("  kalan 4 logda bu topic'lerde ayirt edici imza YOK -> durust kapsam siniri (ag-katmani saldirisi fizige yansimamis)")

UAV Attack — ucus medyanlari (aile imzalari):
              jamming_max  noise_std_max  hdop_max  sats_min  gps_step_max  spd_residual_max  att_missing
label                                                                                                    
gps_jamming          60.0          15.22      4.16       4.0          0.43              2.04          0.0
gps_spoofing          0.0           0.00      0.00      10.0      78379.09         391890.64          0.0
normal                0.0           0.00      0.00      10.0          2.44              0.87          0.0
ping_dos              0.0           0.00      0.00      10.0          2.42              0.56          0.0

Kritik gozlemler:
- gps_jamming: jamming/noise/hdop yuksek, uydu sayisi dusuk -> SINYAL-KALITESI ailesi
- gps_spoofing (SITL): 78 km konum sicramasi -> NAV-BUTUNLUGU ailesi (kaba)
- gps_spoofing (LIVE/hackrf): sicrama YOK (step=0.6 m) ama
  receiver-vs-konum hiz residual'i 5.9 (normal maks ~2) -> analytical redundan

In [4]:
df_a, _, _ = load_source("alfa")
labels_a = manifest["sources"]["alfa"]["flight_labels"]
known = df_a[df_a["label"] != "unknown"]
fl = known.groupby("source_id")
yf = np.array([0 if labels_a[f] == "normal" else 1 for f in fl.groups])
print("ALFA — tek-sinyal ucus-ROC siralamasi (en iyi 8):")
rows = []
for c in ["alt_error_cusum_pos", "alt_error", "aspd_error", "climb_residual", "xtrack_error",
          "roll_error_cusum_pos", "roll_spec_energy_5s", "roll_error_5s_rms", "energy_rate", "turn_residual"]:
    if c not in known.columns:
        continue
    s = fl[c].apply(lambda x: x.abs().max())
    rows.append((c, round(roc_auc_score(yf, s.values), 3)))
print(pd.DataFrame(rows, columns=["sinyal", "ucus_ROC"]).sort_values("ucus_ROC", ascending=False).head(8).to_string(index=False))
print()
print("Kritik gozlem: otopilotun KENDI residual'i (alt_error) uzerinde CUSUM = en guclu tek dedektor.")
print("FableChat'in 'hazir residual + CUSUM' tezi veriyle dogrulandi.")

ALFA — tek-sinyal ucus-ROC siralamasi (en iyi 8):
              sinyal  ucus_ROC
 alt_error_cusum_pos     0.878
      climb_residual     0.806
           alt_error     0.794
        xtrack_error     0.750
roll_error_cusum_pos     0.714
 roll_spec_energy_5s     0.697
         energy_rate     0.669
       turn_residual     0.592

Kritik gozlem: otopilotun KENDI residual'i (alt_error) uzerinde CUSUM = en guclu tek dedektor.
FableChat'in 'hazir residual + CUSUM' tezi veriyle dogrulandi.


## 7. Deney C — Modüler dedektörler + skor füzyonu

Mimari (FableChat bölüm 7):

```
                   ┌─ Kontrol-tepki modülü   (roll/pitch/yaw error + rolling + spektral)
Feature tablosu ───┼─ Rehberlik modülü       (alt_error, aspd_error, xtrack, CUSUM, enerji)   [ALFA]
                   ├─ Nav-bütünlüğü modülü   (gps_step, log_speed, hız residual, ivme)        [UAV]
                   ├─ Sinyal-kalitesi modülü (jamming, noise, DOP, uydu)                       [UAV]
                   └─ Veri-kalitesi modülü   (missingness, stale)                              [UAV]
```

Her modül **kendi dar feature setiyle** bir IF eğitir (sinyal sulanmaz). Uçuş skoru satır skorlarının
**maksimumu** (imza birkaç satırda yaşar; P95 onları kaçırır — 7000 satırlık uçuşta P99 bile 70. satırı seçer).

**Füzyon**: modül skorları farklı ölçeklerde olduğundan her modül kendi val eşiğine oranlanır:

$$R_m(u) = \frac{\max_{t \in u} s_m(x_t)}{\tau_m}, \qquad \tau_m = \max_{u' \in val} \max_{t \in u'} s_m(x_t)$$

$$\text{FüzyonSkoru}(u) = \max_m R_m(u), \qquad \text{alarm} \iff \text{FüzyonSkoru} > 1$$

$\tau_m$ tamamen normal validation'dan gelir; $R_m > 1$ = "bu uçuş, val'deki en kötü normal uçuştan bile uç".

In [5]:
MODULES = {
    "alfa": {
        "kontrol_tepki": ["abs_roll_error", "abs_pitch_error", "abs_yaw_error", "roll_error_rate",
                           "pitch_error_rate", "yaw_error_rate", "roll_error_2s_rms", "roll_error_5s_rms",
                           "pitch_error_2s_rms", "pitch_error_5s_rms", "yaw_error_5s_rms",
                           "roll_spec_energy_5s", "pitch_spec_energy_5s", "roll_error_cusum_pos",
                           "pitch_error_cusum_pos", "turn_residual", "turn_residual_5s_rms"],
        "rehberlik": ["alt_error", "aspd_error", "xtrack_error", "path_dev_mag", "wp_dist",
                       "alt_error_cusum_pos", "alt_error_cusum_neg", "xtrack_error_cusum_pos",
                       "climb_residual", "energy_rate", "altitude_rate", "airspeed_error",
                       "abs_airspeed_error", "xtrack_error_5s_rms"],
    },
    "uav_attack": {
        "nav_butunlugu": ["gps_step_m", "log_gps_speed", "gps_accel_mps2", "vertical_rate_calc",
                           "gps_speed_residual", "vertical_rate_residual", "course_change_deg",
                           "gps_step_5s_max", "gps_step_5s_rms", "gps_speed_residual_cusum_pos",
                           "gps_frozen_count"],
        "sinyal_kalitesi": ["jamming_indicator", "noise_per_ms", "hdop", "vdop", "satellites_used",
                             "s_variance_m_s", "eph", "epv", "jamming_1s_mean", "jamming_5s_max",
                             "noise_5s_std", "noise_5s_mean", "hdop_5s_max", "sats_5s_min",
                             "hdop_cusum_pos", "noise_per_ms_cusum_pos"],
        "veri_kalitesi": ["attitude_missing", "battery_missing", "gps_health_missing",
                           "num_missing_groups", "attitude_stale_count", "attitude_stale_s"],
    },
}

def run_modular(name):
    df, scaler, _ = load_source(name)
    labels = manifest["sources"][name]["flight_labels"]
    if name == "alfa":
        df = df[df["label"] != "unknown"]  # exploration kumesi degerlendirmeye girmez
    scaled = apply_scaler_params(df, scaler)
    results, per_split_flight = [], {}
    for split_name, split in manifest["sources"][name]["splits"].items():
        tr = scaled[scaled["source_id"].isin(split["train"])]
        va = scaled[scaled["source_id"].isin(split["val"])]
        te = scaled[scaled["source_id"].isin(split["test"])]
        ratios = {}
        for mod, cols in MODULES[name].items():
            cols = [c for c in cols if c in scaled.columns]
            m = IsolationForest(n_estimators=300, max_samples=256,
                                random_state=split["seed"], n_jobs=-1).fit(tr[cols])
            va_flight = va.assign(s=anomaly_scores(m, va[cols])).groupby("source_id")["s"].max()
            tau_m = va_flight.max()
            te_flight = te.assign(s=anomaly_scores(m, te[cols])).groupby("source_id")["s"].max()
            ratios[mod] = te_flight / tau_m
        R = pd.DataFrame(ratios)
        R["fusion"] = R.max(axis=1)
        R["label"] = [labels[f] for f in R.index]
        R["y"] = (R["label"] != "normal").astype(int)
        row = {"split": split_name,
               "ucus_roc_fusion": roc_auc_score(R["y"], R["fusion"]),
               "tespit@1": float((R[R["y"] == 1]["fusion"] > 1).mean()),
               "normal_yanlis_alarm@1": float((R[R["y"] == 0]["fusion"] > 1).mean())}
        for mod in MODULES[name]:
            row[f"roc[{mod}]"] = roc_auc_score(R["y"], R[mod])
        results.append(row)
        per_split_flight[split_name] = R
    return pd.DataFrame(results).set_index("split"), per_split_flight

res_a, flights_a = run_modular("alfa")
print("=== ALFA — moduler + fuzyon (ucus duzeyi, 5 seed) ===")
print(res_a.round(3).to_string())
print("\nortalama +- std:")
print((res_a.mean().round(3).astype(str) + " +- " + res_a.std().round(3).astype(str)).to_string())

=== ALFA — moduler + fuzyon (ucus duzeyi, 5 seed) ===
          ucus_roc_fusion  tespit@1  normal_yanlis_alarm@1  roc[kontrol_tepki]  roc[rehberlik]
split                                                                                         
split_00            0.806     0.861                    0.5               0.514           0.806
split_01            0.861     0.722                    0.0               0.556           0.819
split_02            0.556     1.000                    1.0               0.097           0.792
split_03            0.986     0.528                    0.0               0.903           0.958
split_04            0.958     0.778                    0.0               0.486           0.944

ortalama +- std:
ucus_roc_fusion          0.833 +- 0.172
tespit@1                 0.778 +- 0.175
normal_yanlis_alarm@1      0.3 +- 0.447
roc[kontrol_tepki]       0.511 +- 0.286
roc[rehberlik]           0.864 +- 0.081


In [6]:
res_u, flights_u = run_modular("uav_attack")
print("=== UAV ATTACK — moduler + fuzyon (ucus duzeyi, 5 seed) ===")
print(res_u.round(3).to_string())
print("\nortalama +- std:")
print((res_u.mean().round(3).astype(str) + " +- " + res_u.std().round(3).astype(str)).to_string())

=== UAV ATTACK — moduler + fuzyon (ucus duzeyi, 5 seed) ===
          ucus_roc_fusion  tespit@1  normal_yanlis_alarm@1  roc[nav_butunlugu]  roc[sinyal_kalitesi]  roc[veri_kalitesi]
split                                                                                                                   
split_00            0.654     0.308                    0.0               0.462                 0.308               0.615
split_01            0.231     0.923                    1.0               0.615                 0.231               0.923
split_02            0.654     0.308                    0.0               0.462                 0.308               0.615
split_03            0.769     0.538                    0.0               0.615                 0.308               0.615
split_04            0.692     0.769                    1.0               0.615                 0.308               0.615

ortalama +- std:
ucus_roc_fusion            0.6 +- 0.212
tespit@1                 0.569 +- 0

In [7]:
# Tip bazinda tespit (fusion > 1, seed ortalamasi) + hangi modul yakaladi
def per_type(flights):
    rows = []
    for split_name, R in flights.items():
        for lbl, g in R[R["y"] == 1].groupby("label"):
            det = (g["fusion"] > 1)
            which = g.drop(columns=["fusion", "label", "y"]).idxmax(axis=1)
            rows.append({"split": split_name, "label": lbl, "tespit": det.mean(),
                         "baskin_modul": which.mode().iloc[0] if len(which) else "-"})
    t = pd.DataFrame(rows)
    return t.groupby("label").agg(tespit_orani=("tespit", "mean"), baskin_modul=("baskin_modul", lambda x: x.mode().iloc[0]))

print("ALFA tip bazinda:"); print(per_type(flights_a).round(2).to_string())
print("\nUAV Attack tip bazinda:"); print(per_type(flights_u).round(2).to_string())

ALFA tip bazinda:
                      tespit_orani   baskin_modul
label                                            
aileron_fault                 0.77      rehberlik
aileron_rudder_fault          0.20  kontrol_tepki
elevator_fault                0.80      rehberlik
engine_fault                  0.86      rehberlik
rudder_fault                  0.33      rehberlik

UAV Attack tip bazinda:
              tespit_orani     baskin_modul
label                                      
gps_jamming           1.00    veri_kalitesi
gps_spoofing          0.70  sinyal_kalitesi
ping_dos              0.37    veri_kalitesi


## 8. Detection Time — ALFA resmî metrikleri

Rehberlik modülünün satır skorlarıyla, arıza onset'inden ($t_0$ = ilk anomali etiketli satır) ilk alarm satırına
geçen süre. Alarm eşiği: val satır skorlarının Q99'u. $\text{AvgDT} = \overline{t_{alarm} - t_0}$, MaxDT = en kötüsü.

In [8]:
def detection_times(name, module_cols, split_name="split_00"):
    df, scaler, _ = load_source(name)
    labels = manifest["sources"][name]["flight_labels"]
    df = df[df["label"] != "unknown"]
    scaled = apply_scaler_params(df, scaler)
    split = manifest["sources"][name]["splits"][split_name]
    cols = [c for c in module_cols if c in scaled.columns]
    tr = scaled[scaled["source_id"].isin(split["train"])]
    va = scaled[scaled["source_id"].isin(split["val"])]
    m = IsolationForest(n_estimators=300, max_samples=256, random_state=0, n_jobs=-1).fit(tr[cols])
    tau = np.quantile(anomaly_scores(m, va[cols]), 0.99)
    delays, missed = [], 0
    for sid in split["test_anomalous"]:
        g = scaled[scaled["source_id"] == sid].sort_values("t_rel_s")
        g = g.assign(s=anomaly_scores(m, g[cols]))
        onset_rows = g[~g["label"].isin(NORMALS)]
        if onset_rows.empty:
            continue
        onset = onset_rows["t_rel_s"].iloc[0]
        alarms = g[(g["t_rel_s"] >= onset) & (g["s"] > tau)]
        if len(alarms):
            delays.append(alarms["t_rel_s"].iloc[0] - onset)
        else:
            missed += 1
    return delays, missed

delays, missed = detection_times("alfa", MODULES["alfa"]["rehberlik"])
print(f"ALFA rehberlik modulu (split_00): {len(delays)} ucusta alarm, {missed} ucus kacirildi")
print(f"Average Detection Time = {np.mean(delays):.1f} s, Maximum Detection Time = {np.max(delays):.1f} s")
print("(ALFA makalesi kiyas cizgisi: ~senaryoya gore birkac sn - onlarca sn)")

ALFA rehberlik modulu (split_00): 34 ucusta alarm, 2 ucus kacirildi
Average Detection Time = 3.7 s, Maximum Detection Time = 68.1 s
(ALFA makalesi kiyas cizgisi: ~senaryoya gore birkac sn - onlarca sn)


## 9. Leave-dataset-out — UAV-SEAD dış doğrulaması

UAV Attack'ta eğitilen modüler dedektörler + scaler + $\tau_m$'ler, **hiç görülmemiş** UAV-SEAD'e uygulanır.
Asıl metrik: **normal uçuşlarda yanlış alarm** — yeni platformun tüm normallerine "anomali" diyorsa framework
dataset-specific demektir. (Doğru operasyonel kullanım: yeni platformda 10-30 dk normal uçuşla $\tau$ yeniden kalibre edilir.)

In [9]:
df_s, _, _ = load_source("uav_sead")
_, scaler_u, _ = load_source("uav_attack")
labels_s = manifest["sources"]["uav_sead"]["flight_labels"]
df_u2, _, _ = load_source("uav_attack")
scaled_u = apply_scaler_params(df_u2, scaler_u)
scaled_s = apply_scaler_params(df_s, scaler_u)  # BILEREK uav_attack scaler'i: kalibrasyonsuz transfer
split = manifest["sources"]["uav_attack"]["splits"]["split_00"]
tr = scaled_u[scaled_u["source_id"].isin(split["train"])]
va = scaled_u[scaled_u["source_id"].isin(split["val"])]

ratios = {}
for mod, cols in MODULES["uav_attack"].items():
    cols = [c for c in cols if c in scaled_s.columns]
    m = IsolationForest(n_estimators=300, max_samples=256, random_state=0, n_jobs=-1).fit(tr[cols])
    tau_m = va.assign(s=anomaly_scores(m, va[cols])).groupby("source_id")["s"].max().max()
    ratios[mod] = scaled_s.assign(s=anomaly_scores(m, scaled_s[cols])).groupby("source_id")["s"].max() / tau_m
R = pd.DataFrame(ratios); R["fusion"] = R.max(axis=1)
R["label"] = [labels_s[f] for f in R.index]; R["y"] = (R["label"] != "normal").astype(int)
print("SEAD ucus ROC (fusion):", round(roc_auc_score(R["y"], R["fusion"]), 3))
print("Normal ucus yanlis alarm@1:", round(float((R[R["y"] == 0]["fusion"] > 1).mean()), 2))
print("\nSinif bazinda tespit@1:")
print(R[R["y"] == 1].groupby("label")["fusion"].apply(lambda x: (x > 1).mean()).round(2).to_string())
print("\nYORUM: SEAD anomalileri (state-estimation) UAV Attack'in saldiri imzalarindan farkli fizikte --")
print("dusuk transfer BEKLENIR ve 'kalibrasyonsuz transfer siniri' olarak raporlanir. Dogru yol: SEAD")
print("normalleriyle tau kalibrasyonu (split_manifest'te SEAD splitleri hazir).")

SEAD ucus ROC (fusion): 0.375
Normal ucus yanlis alarm@1: 1.0

Sinif bazinda tespit@1:
label
altitude_anomaly             1.0
external_position_anomaly    1.0
global_position_anomaly      1.0
mechanical_fault             1.0

YORUM: SEAD anomalileri (state-estimation) UAV Attack'in saldiri imzalarindan farkli fizikte --
dusuk transfer BEKLENIR ve 'kalibrasyonsuz transfer siniri' olarak raporlanir. Dogru yol: SEAD
normalleriyle tau kalibrasyonu (split_manifest'te SEAD splitleri hazir).


## 10. `anomaly_events` Gold tablosu

Modüler füzyon skorları ortak şemaya yazılır — Dashboard/Kafka `adsb.alerts` beslemesinin temeli.

In [10]:
events, created = [], datetime.now(timezone.utc).isoformat()
for name, flights in [("alfa", flights_a), ("uav_attack", flights_u)]:
    for split_name, R in flights.items():
        for sid, r in R.iterrows():
            events.append({"source_type": name, "source_id": sid,
                           "model_name": "modular_iforest_fusion", "model_version": "ml1_v2",
                           "anomaly_score": float(r["fusion"]), "threshold": 1.0,
                           "is_anomaly": bool(r["fusion"] > 1),
                           "predicted_category": str(r.drop(["fusion", "label", "y"]).idxmax()),
                           "true_label": r["label"], "split_id": split_name,
                           "feature_version": "ml_features_v2", "created_at": created})
events_df = pd.DataFrame(events)
out = ROOT / "data/gold/anomaly_events/anomaly_events.parquet"
out.parent.mkdir(parents=True, exist_ok=True)
events_df.to_parquet(out, index=False); events_df.to_csv(out.with_suffix(".csv"), index=False)
print(f"{len(events_df)} olay -> {out}")
events_df.head(3)

260 olay -> C:\Users\PC_5812_YD26\Desktop\GitHub\AnomalyDetection\data\gold\anomaly_events\anomaly_events.parquet


,source_type,source_id,model_name,model_version,anomaly_score,threshold,is_anomaly,predicted_category,true_label,split_id,feature_version,created_at
0,alfa,carbonZ_2018-07-18-15-53-31_1_engine_failure,modular_iforest_fusion,ml1_v2,1.135359,1.0,True,rehberlik,engine_fault,split_00,ml_features_v2,2026-07-02T13:26:23.242812+00:00
1,alfa,carbonZ_2018-07-18-15-53-31_2_engine_failure,modular_iforest_fusion,ml1_v2,1.043738,1.0,True,rehberlik,engine_fault,split_00,ml_features_v2,2026-07-02T13:26:23.242812+00:00
2,alfa,carbonZ_2018-07-18-16-22-01_engine_failure_wit...,modular_iforest_fusion,ml1_v2,1.110568,1.0,True,rehberlik,engine_fault,split_00,ml_features_v2,2026-07-02T13:26:23.242812+00:00


## 11. Sonuç ve ML-2 yol haritası

**Bu notebook'un kanıtladıkları:**
1. Monolitik satır-bazlı IF bu problemde yapısal olarak zayıf (etiket semantiği + feature sulanması + impute etkisi).
2. Anomali aileleri farklı sinyallerde yaşıyor; **analytical redundancy** (`gps_speed_residual`) gizli/live spoofing'i
   bile yakalıyor; otopilotun kendi residual'ı + CUSUM, ALFA'nın en güçlü tek dedektörü.
3. Modüler IF + val-normalize füzyon, uçuş düzeyinde anlamlı tespit + `predicted_category` (baskın modül) açıklanabilirliği veriyor.
4. Ping DoS'un 4/6 logu bu topic'lerden tespit edilemez → **kapsam beyanına** girer (ağ-katmanı imzası fiziğe yansımamış).

**ML-2 (sıradaki):** 10 sn pencereler (ALFA 40 / UAV 50 örnek), yalnızca normal pencereler,
Dense AE (debug) → LSTM-AE (`LSTM 32 → latent 16 → RepeatVector → LSTM 32`), skor = pencere MSE,
feature-bazlı reconstruction error saklanır. **ML-3 ablation'lar:** missingness ± , airspeed ± ,
residual-only vs raw-only, sentetik enjeksiyon (freeze/bias/drift/stealthy-ramp) test seti.
